# Importing the dataset

In [118]:
import pandas as pd

In [119]:
df=pd.read_csv(r"C:\Users\sekar\Downloads\archive (19)\online_retail_II.csv")
df.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


In [120]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [122]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [124]:
df.dropna(subset=["Customer ID"],inplace=True)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [125]:
print(f"Negative Quantity orders: {(df['Quantity'] < 0).sum()}")

Negative Quantity orders: 18744


In [126]:
cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
len(cancelled)
print(f"Cancelled orders: {(len(cancelled))}")

Cancelled orders: 18744


In [128]:
print(f"Negative/zero Price orders: {(df['Price'] <= 0).sum()}")

Negative/zero Price orders: 71


In [129]:
df = df[df["Quantity"] > 0]
df = df[df["Price"] > 0]

In [130]:
df.shape

(805549, 8)

In [153]:
df["Customer ID"].nunique()

5878

## Dataset Summary

- After cleaning, the dataset contains **5,878 unique customers** across the full 2009–2011 transaction history.

In [131]:
print(df['InvoiceDate'].min(), df['InvoiceDate'].max())

2009-12-01 07:45:00 2011-12-09 12:50:00


## Dataset Overview

- Transaction data spans **December 1, 2009 to December 9, 2011** (~2 years).
- This range was used to define the snapshot date for Recency calculation (max date + 1 day).

In [163]:
df["Invoice_Date"]=df["InvoiceDate"].str.split(" ").str[0]
df["Invoice_Time"]=df["InvoiceDate"].str.split(" ").str[1]
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Invoice_Date,Invoice_Time,Total Price
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12-01,07:45:00,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12-01,07:45:00,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12-01,07:45:00,30.0


In [164]:
pd.to_datetime(df["Invoice_Date"])
df.sort_values(["Invoice_Date"]).groupby(["Customer ID"]).max().head().reset_index()

,Customer ID,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Country,Invoice_Date,Invoice_Time,Total Price
0,12346.0,541431,TEST002,This is a test product.,74215,2011-01-18 10:01:00,7.49,United Kingdom,2011-01-18,13:53:00,77183.6
1,12347.0,581180,85232D,WRAP WEDDING DAY,240,2011-12-07 15:52:00,12.75,Iceland,2011-12-07,15:52:00,249.6
2,12348.0,568172,POST,SWEETIES STICKERS,144,2011-09-25 13:13:00,40.00,Finland,2011-09-25,19:09:00,240.0
3,12349.0,577609,POST,ZINC FOLKART SLEIGH BELLS,48,2011-11-21 09:51:00,300.00,Italy,2011-11-21,13:20:00,300.0
4,12350.0,543037,POST,UNION JACK FLAG PASSPORT COVER,24,2011-02-02 16:01:00,40.00,Norway,2011-02-02,16:01:00,40.0


In [134]:
frequency_df = df.groupby('Customer ID')['Invoice'].nunique().reset_index()
frequency_df.columns = ['Customer ID', 'Frequency']

print(frequency_df.head())

   Customer ID  Frequency
0      12346.0         12
1      12347.0          8
2      12348.0          5
3      12349.0          4
4      12350.0          1


In [135]:
df["Total Price"]=df["Quantity"]*df["Price"]
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Invoice_Date,Invoice_Time,Total Price
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12-01,07:45:00,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12-01,07:45:00,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12-01,07:45:00,30.0


In [136]:
monetary_value_df=df.groupby("Customer ID")["Total Price"].sum().reset_index()
monetary_value_df.head()

,Customer ID,Total Price
0,12346.0,77556.46
1,12347.0,5633.32
2,12348.0,2019.40
3,12349.0,4428.69
4,12350.0,334.40


In [137]:
df['Invoice_Date'] = pd.to_datetime(df['Invoice_Date'])

snapshot_date = df['Invoice_Date'].max() + pd.Timedelta(days=1)

recency_df = df.groupby('Customer ID')['Invoice_Date'].max().reset_index()
recency_df['Recency'] = (snapshot_date - recency_df['Invoice_Date']).dt.days
recency_df = recency_df[['Customer ID', 'Recency']]

display(recency_df.head())

,Customer ID,Recency
0,12346.0,326
1,12347.0,3
2,12348.0,76
3,12349.0,19
4,12350.0,311


In [138]:
df1=pd.merge(recency_df,monetary_value_df,on="Customer ID")

In [139]:
rfm_df=pd.merge(df1,frequency_df,on="Customer ID")
rfm_df.head()

,Customer ID,Recency,Total Price,Frequency
0,12346.0,326,77556.46,12
1,12347.0,3,5633.32,8
2,12348.0,76,2019.40,5
3,12349.0,19,4428.69,4
4,12350.0,311,334.40,1


In [140]:
rfm_df['M_Score'] = pd.qcut(rfm_df['Total Price'], 4, labels=[1, 2, 3, 4])
rfm_df['R_Score'] = pd.qcut(rfm_df['Recency'], 4, labels=[4, 3, 2, 1])
rfm_df['F_Score'] = pd.qcut(rfm_df['Frequency'], 4, labels=False, duplicates='drop') + 1
rfm_df.head()

,Customer ID,Recency,Total Price,Frequency,M_Score,R_Score,F_Score
0,12346.0,326,77556.46,12,4,2,3
1,12347.0,3,5633.32,8,4,4,3
2,12348.0,76,2019.40,5,3,3,2
3,12349.0,19,4428.69,4,4,4,2
4,12350.0,311,334.40,1,1,2,1


In [141]:
rfm_df['RFM_Score'] = rfm_df['R_Score'].astype(int) + rfm_df['F_Score'].astype(int) + rfm_df['M_Score'].astype(int)
display(rfm_df.head())

,Customer ID,Recency,Total Price,Frequency,M_Score,R_Score,F_Score,RFM_Score
0,12346.0,326,77556.46,12,4,2,3,9
1,12347.0,3,5633.32,8,4,4,3,11
2,12348.0,76,2019.40,5,3,3,2,8
3,12349.0,19,4428.69,4,4,4,2,10
4,12350.0,311,334.40,1,1,2,1,4


In [142]:
def segment_customer(RFM_score):
    if RFM_score>=10:
        return "Champions"
    elif RFM_score >= 8:
        return 'Loyal'
    elif RFM_score >= 6:
        return 'At Risk'
    else:
        return 'Lost'

rfm_df["Segment"]=rfm_df["RFM_Score"].apply(segment_customer)
display(rfm_df["Segment"].value_counts())

Segment
Lost         2243
At Risk      1437
Champions    1126
Loyal        1072
Name: count, dtype: int64

- **Lost** is the largest segment (~38% of customers), while **Champions** and **Loyal** together make up ~37%.
- This distribution set the stage for the revenue analysis — despite being the smallest group, Champions turned out to drive the majority of revenue

In [143]:
revenue_contribution=rfm_df.groupby("Segment")["Total Price"].mean().sort_values(ascending=False)
revenue_contribution


Segment
Champions    10770.109951
Loyal         2902.560109
At Risk       1146.881180
Lost           381.931646
Name: Total Price, dtype: float64

- Champions spend **~28x more per customer** than Lost customers, confirming they're the highest-value segment individually, not just in aggregate.
- This average-spend gap reinforces the earlier finding that Champions (19% of customers) drive 71% of total revenue.

In [144]:
segment_revenue_pct = (revenue_contribution / revenue_contribution.sum() * 100).round()
print(segment_revenue_pct)

Segment
Champions    71.0
Loyal        19.0
At Risk       8.0
Lost          3.0
Name: Total Price, dtype: float64


In [147]:
rfm_df.head()

,Customer ID,Recency,Total Price,Frequency,M_Score,R_Score,F_Score,RFM_Score,Segment
0,12346.0,326,77556.46,12,4,2,3,9,Loyal
1,12347.0,3,5633.32,8,4,4,3,11,Champions
2,12348.0,76,2019.40,5,3,3,2,8,Loyal
3,12349.0,19,4428.69,4,4,4,2,10,Champions
4,12350.0,311,334.40,1,1,2,1,4,Lost


In [148]:
country_revenue = df.groupby("Country")["Total Price"].sum().sort_values(ascending=False).reset_index()
display(country_revenue.head(10))

,Country,Total Price
0,United Kingdom,1.472315e+07
1,EIRE,6.216311e+05
2,Netherlands,5.542323e+05
3,Germany,4.312625e+05
4,France,3.552575e+05
5,Australia,1.699681e+05
6,Spain,1.091785e+05
7,Switzerland,1.003653e+05
8,Sweden,9.154972e+04
9,Denmark,6.986219e+04


- The **United Kingdom** dominates revenue overwhelmingly, generating over **23x more** than the next closest market (EIRE).
- International sales are minor by comparison, indicating the business is heavily UK-dependent with limited geographic diversification.

In [151]:
df.groupby("Description")["Total Price"].sum().sort_values(ascending=False).reset_index().head(3)

,Description,Total Price
0,REGENCY CAKESTAND 3 TIER,286486.30
1,WHITE HANGING HEART T-LIGHT HOLDER,252072.46
2,"PAPER CRAFT , LITTLE BIRDIE",168469.60


These three products alone account for a significant share of total revenue, suggesting strong demand for home décor/gift items.


In [152]:
one_time_customers=(rfm_df["Frequency"]==1).sum()/len(rfm_df["Frequency"])*100
print(f"One Time Customers Count :{one_time_customers}")

One Time Customers Count :27.611432460020414


**27.6%** of customers made only a single purchase (`Frequency == 1`).
- This aligns closely with the **38% "Lost"** segment share, suggesting a meaningful portion of one-time buyers never return — a retention/onboarding opportunity.

In [156]:
champions_ids = rfm_df[rfm_df['Segment'] == 'Champions']['Customer ID']
champions_products = df[df['Customer ID'].isin(champions_ids)]

top_champions_products = champions_products.groupby('Description')['Total Price'].sum().sort_values(ascending=False).head(5).reset_index()
display(top_champions_products)

,Description,Total Price
0,REGENCY CAKESTAND 3 TIER,224155.20
1,WHITE HANGING HEART T-LIGHT HOLDER,174089.67
2,JUMBO BAG RED RETROSPOT,111971.42
3,ASSORTED COLOUR BIRD ORNAMENT,95521.38
4,POSTAGE,83614.80


 **REGENCY CAKESTAND 3 TIER** and **WHITE HANGING HEART T-LIGHT HOLDER** are top sellers both overall and within the Champions segment — confirming these are core, high-demand products rather than niche picks.
- Champions alone account for **~78%** of REGENCY CAKESTAND's total revenue (₹2,24,155 of ₹2,86,486), showing how concentrated demand for top products is among high-value customers.


In [170]:
df.groupby("Description")["Quantity"].sum().sort_values(ascending=False).reset_index().head(3)

,Description,Quantity
0,WORLD WAR 2 GLIDERS ASSTD DESIGNS,109169
1,WHITE HANGING HEART T-LIGHT HOLDER,93640
2,"PAPER CRAFT , LITTLE BIRDIE",80995


**WORLD WAR 2 GLIDERS ASSTD DESIGNS** leads in units sold but didn't appear in the top revenue list — likely a low-price, high-volume item.


In [173]:
df.groupby("Country")["Customer ID"].nunique().sort_values(ascending=False).reset_index().head()

,Country,Customer ID
0,United Kingdom,5350
1,Germany,107
2,France,95
3,Spain,41
4,Belgium,29


Out of 5,878 total customers, **~91%** are based in the UK — customer base is even more UK-concentrated than revenue alone suggested.
- Germany and France are distant secondary markets by both customer count and revenue, reinforcing the earlier finding of minimal geographic diversification.